In [64]:
import pandas as pd

df1 = pd.read_csv("../../../etl/raw_data/한국고전종합DB_관계망/itkc_events.csv")
df2 = pd.read_csv("../../../etl/raw_data/한국고전종합DB_관계망/itkc_event_relations.csv")

In [65]:
df1.shape

(1542, 9)

In [66]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 1542 entries, 0 to 1541
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   scope             1542 non-null   str  
 1   event_id          1542 non-null   str  
 2   event_name        1542 non-null   str  
 3   subject_category  1542 non-null   str  
 4   period            1542 non-null   str  
 5   event_date        1532 non-null   str  
 6   person_count      1542 non-null   int64
 7   related_event     605 non-null    str  
 8   detail_url        1542 non-null   str  
dtypes: int64(1), str(8)
memory usage: 108.6 KB


In [67]:
df1.head()

,scope,event_id,event_name,subject_category,period,event_date,person_count,related_event,detail_url
0,event_subject,ITKC_PH_1294A_0435,강동성전투,전쟁,고려,1218년(고종 5) 12월\r\n~ 1219년(고종 6) 1월,2,고려거란전쟁,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
1,event_period,ITKC_PH_1294A_0435,강동성전투,전쟁,고려,1218년(고종 5) 12월\r\n~ 1219년(고종 6) 1월,2,고려거란전쟁,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
2,event_period,ITKC_PH_1294A_0435,강동성전투,전쟁,고려,1218년(고종 5) 12월\r\n~ 1219년(고종 6) 1월,2,고려거란전쟁,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
3,event_period,ITKC_PH_1294A_0436,거란의 항복,전쟁,고려,1219년(고종 6) 1월,2,고려거란전쟁,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
4,event_period,ITKC_PH_1294A_0437,최충헌의 사망,정치인,고려,1219년(고종 6) 9월,4,최씨무인정권,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...


In [68]:
compare_cols = [col for col in df1.columns if col not in ["event_id", "scope"]]

diff_check = (
    df1
    .groupby("event_id")[compare_cols]
    .nunique(dropna=False)
)

diff_rows = diff_check[diff_check.gt(1).any(axis=1)]

diff_rows

,event_name,subject_category,period,event_date,person_count,related_event,detail_url
event_id,,,,,,,
ITKC_PH_1294A_0335,1,1,1,1,1,1,2
ITKC_PH_1294A_0336,1,1,1,1,1,1,2
ITKC_PH_1294A_0337,1,1,1,1,1,1,2
ITKC_PH_1294A_0338,1,1,1,1,1,1,2
ITKC_PH_1294A_0339,1,1,1,1,1,1,2
...,...,...,...,...,...,...,...
ITKC_PH_1305A_0330,1,1,1,1,1,1,2
ITKC_PH_1305A_0331,1,1,1,1,1,1,2
ITKC_PH_1305A_0332,1,1,1,1,1,1,2


In [69]:
event_source_url = (
    df1
    .groupby("event_id")["detail_url"]
    .apply(lambda x: "|".join(sorted(x.dropna().unique())))
    .reset_index(name="source_urls")
)

df1 = (
    df1
    .drop_duplicates(subset=["event_id"])
    .drop(columns=["scope", "person_count", "detail_url"])
    .merge(event_source_url, on="event_id", how="left")
)

In [70]:
df1["event_id"].is_unique

True

In [71]:
df1.duplicated().sum()

np.int64(0)

In [72]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   event_id          600 non-null    str  
 1   event_name        600 non-null    str  
 2   subject_category  600 non-null    str  
 3   period            600 non-null    str  
 4   event_date        595 non-null    str  
 5   related_event     224 non-null    str  
 6   source_urls       600 non-null    str  
dtypes: str(7)
memory usage: 32.9 KB


In [73]:
df1.shape

(600, 7)

In [74]:
df2.shape

(15392, 10)

In [75]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 15392 entries, 0 to 15391
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   scope               15392 non-null  str    
 1   event_id            15392 non-null  str    
 2   event_name          15392 non-null  str    
 3   relation_type       15392 non-null  str    
 4   person_id           15392 non-null  str    
 5   person_name         15392 non-null  str    
 6   related_event_id    0 non-null      float64
 7   related_event_name  0 non-null      float64
 8   evidence_url        0 non-null      float64
 9   detail_url          15392 non-null  str    
dtypes: float64(3), str(7)
memory usage: 1.2 MB


In [76]:
df2.duplicated().sum()

np.int64(1556)

In [77]:
df2.head(5)

,scope,event_id,event_name,relation_type,person_id,person_name,related_event_id,related_event_name,evidence_url,detail_url
0,event_subject,ITKC_PH_1294A_0435,강동성전투,사건인물,P009535,김인경(金仁鏡),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
1,event_subject,ITKC_PH_1294A_0435,강동성전투,사건인물,P058283,조충(趙冲),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
2,event_period,ITKC_PH_1294A_0435,강동성전투,사건인물,P009535,김인경(金仁鏡),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
3,event_period,ITKC_PH_1294A_0435,강동성전투,사건인물,P058283,조충(趙冲),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
4,event_period,ITKC_PH_1294A_0435,강동성전투,사건인물,P009535,김인경(金仁鏡),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...


In [78]:
df2['related_event_id'].nunique()

0

In [79]:
df2['evidence_url'].nunique()

0

In [80]:
df2 = df2.drop_duplicates(subset=["event_id", "person_id"])

In [81]:
df2.info()

<class 'pandas.DataFrame'>
Index: 6918 entries, 0 to 14476
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   scope               6918 non-null   str    
 1   event_id            6918 non-null   str    
 2   event_name          6918 non-null   str    
 3   relation_type       6918 non-null   str    
 4   person_id           6918 non-null   str    
 5   person_name         6918 non-null   str    
 6   related_event_id    0 non-null      float64
 7   related_event_name  0 non-null      float64
 8   evidence_url        0 non-null      float64
 9   detail_url          6918 non-null   str    
dtypes: float64(3), str(7)
memory usage: 594.5 KB


In [82]:
df2.tail()

,scope,event_id,event_name,relation_type,person_id,person_name,related_event_id,related_event_name,evidence_url,detail_url
14472,event_subject,ITKC_PH_1305A_0334,심수경의 과거 동기,사건인물,P056339,조보(趙溥),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
14473,event_subject,ITKC_PH_1305A_0334,심수경의 과거 동기,사건인물,P058193,조징(趙澄),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
14474,event_subject,ITKC_PH_1305A_0334,심수경의 과거 동기,사건인물,P066169,홍천민(洪天民),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
14475,event_subject,ITKC_PH_1305A_0334,심수경의 과거 동기,사건인물,P066576,황림(黃琳),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...
14476,event_subject,ITKC_PH_1305A_0334,심수경의 과거 동기,사건인물,P066959,황응규(黃應奎),NaN,NaN,NaN,https://db.itkc.or.kr/people/viewEvnt?gubun=ev...


In [83]:
df2['relation_type'].unique()

<StringArray>
['사건인물']
Length: 1, dtype: str

In [84]:
df2 = df2.drop(columns=['scope', 'detail_url', 'evidence_url', 'related_event_id', 'related_event_name','relation_type'])

In [85]:
df2.head()

,event_id,event_name,person_id,person_name
0,ITKC_PH_1294A_0435,강동성전투,P009535,김인경(金仁鏡)
1,ITKC_PH_1294A_0435,강동성전투,P058283,조충(趙冲)
6,ITKC_PH_1294A_0436,거란의 항복,P011067,김취려(金就礪)
7,ITKC_PH_1294A_0436,거란의 항복,P058283,조충(趙冲)
8,ITKC_PH_1294A_0437,최충헌의 사망,P008037,김약선(金若先)


In [86]:
df1['subject_category'].nunique()

119

In [88]:
df1['subject_category'].unique()

<StringArray>
[               '전쟁',               '정치인',                '민란',
                '반란',    '반란,\r\n\r\n정치인',   '정치일반,\r\n\r\n국왕',
                '국왕',                '세자',              '인물기타',
     '반란,\r\n\r\n국왕',
 ...
                 '청',   '정치일반,\r\n\r\n붕당',  '옥사,\r\n\r\n고변/탄핵',
                '예송',     '예송,\r\n\r\n비빈',  '정치일반,\r\n\r\n정치인',
     '세자,\r\n\r\n비빈', '정치일반,\r\n\r\n묘정배향',     '옥사,\r\n\r\n세자',
              '사가독서']
Length: 119, dtype: str

## EDA 정리: event 데이터 중복과 `scope` 해석

현재 `df1`은 `itkc_events.csv`에서 읽은 사건 기본 정보이고, `df2`는 `itkc_event_relations.csv`에서 읽은 사건-인물 관계 정보다.

### 1. `scope` 해석

`df1.scope`는 최종 `Event` 노드의 역사 속성이라기보다, 같은 사건이 어떤 수집 경로에서 발견되었는지를 나타내는 값으로 본다.

```text
event_subject : 주제/분류 기준 수집 경로
event_period  : 시대 기준 수집 경로
```

현재 분포는 다음처럼 600개씩 동일하다.

```text
event_subject    600
event_period     600
```

즉 같은 사건이 `event_subject`, `event_period` 두 경로에서 중복 수집된 구조로 보인다.

### 2. `event_id` 기준 중복 제거 가능 여부 확인

`event_id`만 보고 바로 중복 제거하면 정보가 사라질 수 있으므로, 먼저 같은 `event_id` 안에서 `scope` 외의 값이 달라지는지 확인한다.

```python
compare_cols = [col for col in df1.columns if col not in ["event_id", "scope"]]

diff_check = (
    df1
    .groupby("event_id")[compare_cols]
    .nunique(dropna=False)
)

diff_rows = diff_check[diff_check.gt(1).any(axis=1)]
diff_rows
```

이 코드의 의미는 다음과 같다.

- `event_id`, `scope`를 제외한 컬럼만 비교한다.
- `event_id`별로 각 컬럼의 고유값 개수를 센다.
- 어떤 컬럼이라도 고유값 개수가 2개 이상이면 `diff_rows`에 남긴다.
- `diff_rows`가 비어 있으면 같은 `event_id` 안에서 다른 값은 `scope`뿐이라는 뜻이다.

현재 확인 결과는 `event_name`, `subject_category`, `period`, `event_date`, `person_count`, `related_event`는 모두 고유값 개수가 `1`이고, `detail_url`만 고유값 개수가 `2`로 보인다.

즉 같은 `event_id` 안에서 사건 내용 자체는 동일하고, 수집 경로별 상세 URL만 다르다. 따라서 `event_id` 기준으로 Event를 1개만 남기되, `detail_url`은 버리기보다 여러 출처 URL을 `source_urls`로 합쳐 보존하는 방식이 적절하다.

`df1.drop_duplicates(subset=["event_id"])`와 `df1.drop_duplicates(subset=["event_id", "detail_url"])`의 결과 수가 다른 이유도 이 때문이다. 같은 사건이라도 `detail_url`이 2개라서, `event_id + detail_url` 기준으로 보면 서로 다른 행으로 남는다.

### 3. 최종 Event 노드 컬럼 판단

최종 `Event` 노드에는 `scope`를 넣지 않는다. `scope`는 수집 경로 메타데이터이고, 사건 자체의 속성은 아니기 때문이다.

1차 Event 노드 기준 컬럼 판단은 다음과 같다.

| 컬럼 | 의미 | 처리 | 사용 방식/이유 |
|---|---|---|---|
| `scope` | 수집 경로 | 제외 | `event_subject`, `event_period`는 사건 속성이 아니라 수집 route라서 최종 Event 노드에는 넣지 않는다. |
| `event_id` | 사건 고유 ID | 사용 | Event 노드의 primary key 역할. 중복 제거 기준도 `event_id`로 둔다. |
| `event_name` | 사건명 | 사용 | Event 노드의 `name` 속성으로 사용한다. |
| `subject_category` | 사건 주제 분류 원문 | 사용 | 바로 표준 Category로 합치지 않고, `EventCategory` 또는 매핑 후보로 사용한다. |
| `period` | 시대 원문 | 사용 | `Period` 노드 연결 후보로 사용하되, 원문 값도 보존한다. |
| `event_date` | 날짜/기간 원문 | 사용 | 연도, 월, 왕대 파싱 후보로 사용한다. 파싱 전에는 원문 속성으로 보존한다. |
| `person_count` | 관련 인물 수 | 제외 가능 | `df2`의 `event_id`, `person_id` 관계에서 다시 계산할 수 있으므로 원본 count에 의존하지 않는다. |
| `related_event` | 관련 사건명/사건 묶음 | 사용 | 같은 전쟁, 옥사, 사건군을 묶는 `EventGroup` 후보로 사용한다. |
| `detail_url` | 상세 페이지 URL | `source_urls`로 보존 | 같은 `event_id`에 URL이 2개 있으므로 단일 값으로 고르지 않고 합쳐서 출처 추적용으로 둔다. |

추천 정리 코드는 다음과 같다.

```python
event_source_url = (
    df1
    .groupby("event_id")["detail_url"]
    .apply(lambda x: "|".join(sorted(x.dropna().unique())))
    .reset_index(name="source_urls")
)

event_df = (
    df1
    .drop_duplicates(subset=["event_id"])
    .drop(columns=["scope", "person_count", "detail_url"])
    .merge(event_source_url, on="event_id", how="left")
)
```

`detail_url`을 그대로 남기면 같은 사건에 대해 어떤 URL을 대표값으로 삼을지 애매해진다. 그래서 `source_urls`로 합친 뒤 Event 노드의 출처 속성으로 쓰는 편이 낫다.

### 4. `subject_category` 후속 처리

`subject_category`는 현재 고유값이 119개이고, 아래처럼 복수 분류가 문자열 안에 같이 들어간 경우가 있다.

```text
반란,\r\n\r\n정치인
정치일반,\r\n\r\n국왕
옥사,\r\n\r\n고변/탄핵
```

따라서 `event.csv.subject_category`는 바로 표준 카테고리로 쓰지 말고, 별도 `category_mapping.csv`에서 표준 `Category`로 매핑한다.

```text
event.csv.subject_category -> category_mapping.csv -> Category
```

즉, 이벤트 카테고리는 `history_terms.term_lk`와 직접 합치는 것이 아니라 공통 표준 카테고리 사전에 매핑하는 방식으로 처리한다.

### 5. Event 데이터의 Neo4j 사용 방향

현재 event 쪽에서 바로 만들 수 있는 데이터는 다음과 같다.

| 만들 데이터 | 주로 쓰는 컬럼 | 설명 |
|---|---|---|
| `Event` 노드 | `event_id`, `event_name`, `event_date`, `period`, `source_urls` | 사건의 기본 노드. 날짜와 시대는 파싱 전 원문도 보존한다. |
| `EventCategory` 후보 | `subject_category` | 119개 원문 분류를 토큰화해서 이벤트 전용 카테고리 사전으로 만든다. |
| `Event - HAS_EVENT_CATEGORY - EventCategory` | `event_id`, `subject_category` | 사건과 이벤트 분류의 직접 관계. |
| `EventCategory - MAPPED_TO - Category` 후보 | `subject_category`, `history_terms.term_lk` 기반 category dictionary | 이벤트 분류와 역사용어 표준 카테고리는 매핑표로 연결한다. |
| `Event - IN_PERIOD - Period` 후보 | `period`, `event_date` | `period`는 시대명, `event_date`는 상세 날짜/기간 파싱 후보로 사용한다. |
| `Event - PART_OF_EVENT_GROUP - EventGroup` 후보 | `related_event` | `고려거란전쟁`처럼 여러 사건을 묶는 상위 사건군 후보로 사용한다. |
| `Person - INVOLVED_IN - Event` 관계 | `df2.event_id`, `df2.person_id`, `relation_type` | 인물 관계는 `df1.person_count`가 아니라 `df2`에서 만든다. |

따라서 event 쪽 1차 결론은 다음과 같다.

- `scope`, `person_count`는 최종 Event 노드에서 제외한다.
- `detail_url`은 드랍하지 않고 `source_urls`로 묶어 출처 속성으로 보존한다.
- `subject_category`는 바로 `history_terms` 카테고리와 합치지 않고, 이벤트 카테고리 사전과 매핑표를 만든다.
- `period`, `event_date`는 아직 파싱하지 말고 원문을 보존한 뒤, 기간/왕대 사전이 생기면 `Period`와 연결한다.
- `related_event`는 단순 문자열 속성으로만 끝내지 말고 `EventGroup` 후보로 본다.
- person 관계는 `person_relations.csv` 또는 `itkc_event_relations.csv`를 파악한 뒤 최종 관계 사전을 만든다.
